# NSR 전사 서버 — 구글 콜랩판

폰 대신 콜랩의 무료 GPU가 전사합니다. 폰보다 수십 배 빠릅니다.

**쓰는 법 (처음 3분)**

1. 위 메뉴 **런타임 → 런타임 유형 변경**에서 **T4 GPU** 를 고르십시오.
2. **런타임 → 모두 실행**을 누르십시오. 첫 실행은 모델을 받느라 2~5분 걸립니다.
3. 마지막 셀 출력에 나오는 **주소를 복사**해서, NSR 앱의
   **설정 → 전사 → 전사 서버·모델**의 **주소 칸**에 붙여넣으십시오.
4. **전사 모델은 앱에서 고릅니다** — 앱의 '전사 모델' 목록에서 누르면
   다음 전사부터 이 서버가 그 모델로 갈아끼워 돌립니다. 여기 '기본 모델' 셀의
   선택 상자는 앱이 아무것도 고르지 않았을 때만 쓰입니다.

그 다음부터는 앱에서 평소처럼 전사를 누르면 콜랩이 대신 일합니다.
30분 조각 기준 대략 1~3분입니다(추정 — 세션마다 다릅니다). 앱에서 처음
고르는 모델은 내려받느라 몇 분 더 걸립니다(그 세션에서 한 번만).
**마지막 셀이 계속 실행 중으로 보이는 것이 정상입니다** — 거기에 전사
접수/완료 로그가 실시간으로 찍힙니다.

**알고 쓰십시오**

- 기록 음성 **원본이 구글(콜랩) 서버와 Cloudflare 터널을 지나갑니다.**
  내 컴퓨터가 아닙니다. 이 경로가 싫으면 내 컴퓨터 서버(앱의 '내 컴퓨터' 모드)를 쓰십시오.
- 주소 끝에 무작위 비밀 문자열이 붙어 있어서, 주소를 통째로 모르는 남은 못 씁니다.
  그래도 주소를 다른 곳에 붙여넣지 마십시오.
- 콜랩 화면에 **'런타임 연결이 끊겼습니다'가 떠도 전사는 계속되고 있을 수 있습니다** —
  브라우저와 콜랩 사이의 연결과, 폰과 전사 서버 사이의 터널은 별개입니다.
  폰의 진행률이 오르고 있으면 서버는 살아 있는 것입니다. '재연결'은 눌러도 되지만,
  전사가 도는 동안 '모두 실행'을 다시 하지는 마십시오. 세션이 정말 회수되더라도
  앱은 그때까지 받은 부분을 저장해 두고, 기록은 다시 전사할 수 있게 남습니다.
- 콜랩 무료 세션은 탭을 닫거나 오래 놔두면 꺼집니다. **전사하는 동안 이 탭을
  열어 두십시오.** 꺼졌으면 '모두 실행'을 다시 — 주소가 새로 나오니 앱에도 다시 넣습니다.
- 전사할 때만 켜는 개인용입니다. 상시 서버로 두는 것은 콜랩 이용 규칙과 맞지 않습니다.
- 이 노트가 고쳐지면 **위 깃허브 링크로 새로 열어야** 최신판입니다.
  드라이브에 저장해 둔 사본은 옛 판 그대로입니다.


In [ ]:
# 필요한 것 설치 + 터널 프로그램 받기 (1~2분)
# nvidia-cudnn/cublas 를 같이 까는 이유: 콜랩 기본 환경의 cuDNN 판이
# faster-whisper(ctranslate2)와 어긋나면 첫 전사에서 파이썬이 통째로
# 죽는다("kernel restarted") — 실사용에서 그대로 재현된 사고다.
%pip -q install faster-whisper fastapi uvicorn python-multipart nvidia-cudnn-cu12 nvidia-cublas-cu12
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print("설치 끝. 다음 셀로.")


In [ ]:
# ── 기본 모델 (앱이 고르지 않았을 때만) ──────────────────
# 모델은 보통 **앱의 '전사 모델' 목록에서** 고릅니다 — 앱이 전사 요청에
# 모델 id 를 실어 보내면 이 서버가 그 모델로 갈아끼워 돌립니다.
# 여기 선택 상자는 앱이 아무것도 고르지 않았을 때의 기본값입니다.
# 목록에 없는 공개 CT2 형식 모델 id 를 직접 붙여넣어도 됩니다.
#
#   NSR 한국어 Medium (대화 특화)       기본. 한국어 대화 1,273시간(소음 대화
#                                       포함)으로 재학습된 medium. 우리 저장소
#                                       에서 받는다(1.5GB, 램 안전, 허깅페이스
#                                       안 거침).
#   ghost613/...-large-v3-turbo-korean  한국어 turbo(낭독 음성 학습). 더 크고
#                                       무료 콜랩에서 램이 빠듯할 수 있다.
#   deepdml/...-large-v3-turbo-ct2      다국어 turbo. 빠르고 램 안전(1.6GB).
#   Systran/faster-whisper-large-v3     다국어 large. 제일 정확, 제일 느림(3GB).
#   Systran/faster-whisper-medium       다국어 medium. 가볍고 빠름.
MODEL_ID = "NSR 한국어 Medium (대화 특화)"  #@param ["NSR 한국어 Medium (대화 특화)", "ghost613/faster-whisper-large-v3-turbo-korean", "deepdml/faster-whisper-large-v3-turbo-ct2", "Systran/faster-whisper-large-v3", "Systran/faster-whisper-medium"] {allow-input: true}
print(f"이 세션의 기본 전사 모델: {MODEL_ID} (앱에서 고르면 그쪽이 우선합니다)")


In [ ]:
# 전사 서버 — 접수하고(202) 뒤에서 돌리고, 앱이 몇 초마다 결과를 물어간다.
#
# 왜 비동기인가: 다 될 때까지 한 요청으로 기다리는 방식은 앱의 업로드
# 클라이언트(60초)와 Cloudflare 터널(약 100초)이 먼저 끊는다 — 실기기
# 타임아웃으로 재현된 사실이다. 긴 기록일수록 전사가 몇 분씩 걸리므로
# "접수증(job_id) → 진행 조회 → 완성본 수령" 구조여야 한다.
#
# 세그먼트는 다 되고 나서가 아니라 **되는 족족** 조회 응답에 실어 준다
# (?since=N 증분). 결과가 세션 메모리에만 있으면, 100%까지 가 놓고 세션이
# 회수되는 순간 전부 버려진다 — 실사용 사고다. 앱이 조금씩 받아 두면
# 세션이 죽어도 받은 데까지는 폰에 남는다.
#
# 모델은 요청이 정한다: 앱이 model 파라미터로 보낸 id 를 get_model 이
# 게으르게 실어 준다(다르면 갈아끼운다). 앱이 안 보내면 세션 기본값.
import os
import tempfile
import threading
import traceback
import uuid

from fastapi import FastAPI, File, Form, UploadFile
from fastapi.responses import JSONResponse


def build_app(get_model, secret: str, log=print) -> FastAPI:
    # log: 백그라운드 스레드의 print 는 셀이 끝난 뒤에는 콜랩 화면에 안
    # 보인다. 실행 셀이 큐를 비우며 대신 찍도록 콜백으로 받는다.
    app = FastAPI()
    jobs = {}
    gpu_lock = threading.Lock()  # GPU 는 하나 — 작업을 줄 세운다.

    def run_job(job_id, path, language, temperature, prompt, requested_model):
        job = jobs[job_id]
        try:
            with gpu_lock:
                # 모델 준비(첫 사용이면 다운로드)는 몇 분 걸릴 수 있다.
                # stage 로 알려 앱이 "멈췄나?" 대신 "준비 중"을 보여주게 한다.
                job["status"] = "processing"
                job["stage"] = "model"
                model = get_model(requested_model)
                job["stage"] = "transcribe"
                segment_iter, info = model.transcribe(
                    path,
                    language=language or "ko",
                    temperature=temperature,
                    initial_prompt=prompt or None,
                    beam_size=5,
                    # 잡음·무음 구간에서 같은 문장이 반복되는 환각을 줄인다.
                    condition_on_previous_text=False,
                )
                segments = job["segments"]  # 조회 응답이 같은 목록을 증분으로 내준다.
                for i, s in enumerate(segment_iter):
                    segments.append(
                        {"id": i, "start": round(s.start, 2), "end": round(s.end, 2), "text": s.text}
                    )
                    if info.duration:
                        job["progress"] = min(s.end / info.duration, 1.0)
                job["result"] = {
                    "task": "transcribe",
                    "language": info.language,
                    "duration": round(info.duration, 2),
                    "text": "".join(s["text"] for s in segments).strip(),
                    "segments": segments,
                }
                job["status"] = "done"
                log(f"전사 끝({job_id[:8]}): {len(segments)}문장 / {round(info.duration)}초")
        except Exception:
            trace = traceback.format_exc()
            log(trace)
            job["error"] = trace[-1500:]
            job["status"] = "error"
        finally:
            os.unlink(path)

    @app.get(f"/{secret}/health")
    def health():
        return {"status": "ok"}

    @app.post(f"/{secret}/v1/audio/transcriptions")
    async def transcribe(
        file: UploadFile = File(...),
        language: str = Form("ko"),
        temperature: float = Form(0.0),
        prompt: str = Form(""),
        response_format: str = Form("verbose_json"),
        model_name: str = Form("", alias="model"),  # 앱의 모델 선택 — 비면 세션 기본값
    ):
        suffix = os.path.splitext(file.filename or "audio.m4a")[1] or ".m4a"
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
            f.write(await file.read())
            path = f.name
        job_id = uuid.uuid4().hex
        jobs[job_id] = {"status": "queued", "progress": 0.0, "stage": "queued", "segments": []}
        threading.Thread(
            target=run_job,
            args=(job_id, path, language, temperature, prompt, model_name),
            daemon=True,
        ).start()
        log(f"전사 접수({job_id[:8]}): {file.filename}" + (f" · 모델 {model_name}" if model_name else ""))
        return JSONResponse(status_code=202, content={"job_id": job_id, "status": "queued"})

    @app.get(f"/{secret}/v1/audio/transcriptions/{{job_id}}")
    def job_status(job_id: str, since: int = 0):
        job = jobs.get(job_id)
        if job is None:
            return JSONResponse(
                status_code=404,
                content={"error": "모르는 작업입니다. 콜랩 세션이 재시작됐으면 전사를 다시 시작하십시오."},
            )
        if job["status"] == "done":
            return {"status": "done", "result": job["result"]}
        if job["status"] == "error":
            return {"status": "error", "error": job["error"]}
        done = job["segments"]
        return {
            "status": job["status"],
            "progress": round(job.get("progress", 0.0), 3),
            "stage": job.get("stage"),
            # since 이후의 새 세그먼트만 — 3초마다 물어도 응답이 가볍다.
            "segments": done[max(since, 0):],
            "next": len(done),
        }

    return app


In [ ]:
# 모델을 싣고 서버·터널을 띄운다. 마지막에 나오는 주소를 앱에 넣으면 된다.
import ctypes
import gc
import glob
import os
import re
import secrets
import subprocess
import tarfile
import threading
import time
import urllib.request

import numpy as np
import psutil

# cuDNN/cuBLAS 를 먼저 손으로 적재한다. 콜랩 기본 환경의 판과 어긋나면
# 첫 전사에서 파이썬이 통째로 죽는데("kernel restarted", 추적도 안 남는다),
# 방금 설치한 판을 절대 경로로 미리 올려 두면 이후 탐색이 이쪽을 쓴다.
for pattern in (
    "/usr/local/lib/python3*/dist-packages/nvidia/cublas/lib/libcublas*.so*",
    "/usr/local/lib/python3*/dist-packages/nvidia/cudnn/lib/libcudnn*.so*",
):
    for lib in sorted(glob.glob(pattern)):
        try:
            ctypes.CDLL(lib)
        except OSError:
            pass

import ctranslate2
import uvicorn
from faster_whisper import WhisperModel
from huggingface_hub import snapshot_download

PORT = 8000

gpu = ctranslate2.get_cuda_device_count() > 0
if not gpu:
    print("⚠ GPU 가 안 잡혔습니다. 런타임 → 런타임 유형 변경 → T4 GPU 를 고른 뒤")
    print("  '모두 실행'을 다시 하십시오. CPU 로도 되지만 몇 배 느립니다.")

# 백그라운드 스레드의 print 는 셀이 끝나면 화면에 안 보인다. 큐에 쌓고
# 아래 상주 루프가 대신 찍는다 — 서버가 뜨기 전(초기 적재)에는 바로 찍는다.
events = []
serving = threading.Event()

def say(msg):
    if serving.is_set():
        events.append(msg)
    else:
        print(msg, flush=True)

# ── 모델 준비 — 앱이 고른 모델을 게으르게 싣고, 다르면 갈아끼운다 ──
# 지난 사고들을 여기서 계속 막는다.
#  · 401 사고: 없는 저장소에 익명 요청이 가면 허깅페이스는 404 대신
#    401("Invalid username or password")을 준다. token=False 로 못박아
#    콜랩 '보안 비밀'의 낡은 HF_TOKEN 이 끼어드는 사고도 함께 잠갔다.
#  · 램 사고: compute_type 이 저장 형식과 다르면 적재 중 변환 복사로 램
#    정점이 두 배가 돼 무료 콜랩(12.7GB)을 넘겼다. 저장 형식 그대로 싣고,
#    모델을 갈아끼울 때는 **이전 모델을 먼저 내려놓고** 받는다 — 두 모델이
#    동시에 램에 있는 순간을 만들지 않는다.
NSR_ID = "nsr-korean-medium"
NSR_MIRROR = ("https://github.com/lulus-cat/NSR-project/releases/download/models/"
              "ct2-korean-medium-1273h-fp16.tar.gz")


def canonical(mid):
    mid = (mid or "").strip()
    if not mid:
        return None
    if mid == NSR_ID or mid.startswith("NSR"):
        return NSR_ID
    return mid


def fetch_model_dir(mid):
    """모델 파일을 마련하고 (경로, compute_type) 을 돌려준다."""
    if mid == NSR_ID:
        # 기본: 한국어 대화 1,273시간 재학습 medium 의 float16 CT2 변환본.
        # 우리 Releases 미러라 허깅페이스를 아예 안 거친다.
        model_dir = "/content/nsr-korean-medium"
        if not os.path.exists(os.path.join(model_dir, "model.bin")):
            say("한국어 Medium(대화 특화)을 받는 중 — 약 1.4GB, 1~3분...")
            done = [-1]
            def _pct(count, block, total):
                pct = min(count * block * 100 // max(total, 1), 100)
                if pct // 10 > done[0]:
                    done[0] = pct // 10
                    say(f"  ...{pct}%")
            tar_path, _ = urllib.request.urlretrieve(NSR_MIRROR, reporthook=_pct)
            os.makedirs(model_dir, exist_ok=True)
            with tarfile.open(tar_path) as tar:
                tar.extractall(model_dir, filter="data")
            os.remove(tar_path)
        return model_dir, ("float16" if gpu else "int8")

    compute = "float16" if gpu else "int8"
    if "korean" in mid.lower():
        # float32 로 저장된 파인튜닝판 — 변환 없이 그대로("default") 싣는다.
        compute = "default" if gpu else "int8"
        say("한국어 파인튜닝판: 변환 없이 그대로 싣습니다(램 절약). 무료 콜랩에서")
        say("빠듯할 수 있습니다 — 세션이 죽으면 기본 모델로 돌아가십시오.")
    say(f"모델 받는 중: {mid} (이 세션에서 처음 쓸 때만 내려받습니다)")
    try:
        model_dir = snapshot_download(
            mid,
            token=False,
            allow_patterns=["config.json", "preprocessor_config.json", "model.bin",
                            "tokenizer.json", "vocabulary.*"],
        )
    except Exception as e:
        raise RuntimeError(
            f"모델을 못 받았습니다: {mid} — id 철자를 확인하십시오"
            " (공개된 CT2 형식 모델이어야 합니다. 401 은 대부분 없는 저장소라는 뜻입니다)."
        ) from e
    return model_dir, compute


_current = {"id": None, "model": None}
_model_lock = threading.Lock()


def get_model(requested=None):
    """앱이 요청한 모델을 돌려준다. 안 고르면 위 셀의 기본값."""
    mid = canonical(requested) or canonical(MODEL_ID) or NSR_ID
    with _model_lock:
        if _current["id"] == mid:
            return _current["model"]
        if _current["model"] is not None:
            say(f"모델 교체: {_current['id']} → {mid}")
            _current["model"] = None
            _current["id"] = None
            gc.collect()  # 이전 모델의 램·VRAM 을 먼저 돌려받는다.
        model_dir, compute = fetch_model_dir(mid)
        say(f"모델 여는 중: {mid}")
        model = WhisperModel(model_dir, device="cuda" if gpu else "cpu", compute_type=compute)
        _current["id"] = mid
        _current["model"] = model
        return model


# 자가 시험: 주소를 내주기 전에 기본 모델로 1초짜리 무음을 전사해 본다.
# GPU 경로가 죽을 거라면 앱이 기다리다 타임아웃 나는 대신 여기서 바로 죽어
# 원인이 이 셀에 보인다. 통과하면 실전도 같은 경로다.
print("자가 전사 시험 중...")
list(get_model().transcribe(np.zeros(16000, dtype=np.float32), language="ko")[0])
print("자가 전사 시험 통과 — 전사 경로 정상.")
vm = psutil.virtual_memory()
print(f"메모리 {vm.used / 1e9:.1f} / {vm.total / 1e9:.1f} GB 사용 중 — 10GB 를 넘어가면 위험하다.")

secret = secrets.token_urlsafe(12)
app = build_app(get_model, secret, log=say)
threading.Thread(
    target=lambda: uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning"),
    daemon=True,
).start()

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
url = None
deadline = time.time() + 60
while time.time() < deadline and url is None:
    line = tunnel.stdout.readline()
    if not line and tunnel.poll() is not None:
        break
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line or "")
    if m:
        url = m.group(0)
if not url:
    raise RuntimeError("터널 주소를 못 받았습니다. 이 셀만 한 번 더 실행해 보십시오.")

print()
print("=" * 62)
print("NSR 앱에 넣을 주소 — 설정 → 전사 → 전사 서버·모델의 주소 칸:")
print()
print(f"    {url}/{secret}")
print()
print("전사 모델은 앱의 '전사 모델' 목록에서 고르십시오. 이 탭을 닫으면 서버도 꺼집니다.")
print("=" * 62)
print()
print("이 셀은 계속 실행 중인 것이 정상입니다 — 전사 접수/완료 로그가 아래에 찍힙니다.")
serving.set()
while True:
    time.sleep(2)
    while events:
        print(events.pop(0), flush=True)
